> In this notebook we upload our raw cs files and preprocess them before the final pipeline integration.

> Then after preprocessing, each of these datasets are transformed into parquet files for more efficient operations.

In [7]:
# We'll first have to create training, validation, and test dataset with same distribution.
import pandas as pd
import numpy as np
import re
import re
import numpy as np
import pandas as pd
from pathlib import Path
from rapidfuzz import fuzz
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
# I wanna see as much content as possible


amazon= pd.read_csv("raw/df_amazon.csv",index_col=0,low_memory=False)
goodreads= pd.read_csv("raw/df_goodreads.csv",index_col=0)
goodreads=goodreads[["isbn","title","author","rating","numRatings","language","genres","bookFormat","edition","pages","publisher","publishDate","price","ISBN_clean"]]
metabooks= pd.read_csv("raw/df_meta_books_sampled.csv",index_col=0)
metabooks=metabooks[["ISBN_10","title","Author_Name","average_rating","rating_number","Language","categories","Publisher","published_date","Page_Count","price","ISBN_clean"]].reset_index(drop=True)

### Transformation for Amazon

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

def to_nullable_string(s: pd.Series) -> pd.Series:
    # Normalize to pandas StringDtype, stripping whitespace; empty → <NA>
    return (
        s.astype("string")
         .str.strip()
         .replace({"": pd.NA})
    )

def coerce_year_to_int16(s: pd.Series) -> pd.Series:
    # Keep only 4-digit numeric years; non-numeric → <NA>
    y = pd.to_numeric(s, errors="coerce")
    # If someone passed floats-as-strings like "2001.0", coerce to int where safe
    y = y.where(y.isna(), y.astype("Int64"))
    # Bound-check (optional): keep plausible years
    y = y.where((y.isna()) | ((y >= 0) & (y <= 32767)), pd.NA)
    return y.astype("Int16")

def process_amazon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    required = {"title", "Book-Author", "Year-Of-Publication", "Publisher"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing expected columns: {sorted(missing)}")

    # Normalize text columns to pandas 'string' dtype
    text_cols = [ "title", "Book-Author", "Publisher"]
    for c in text_cols:
        df[c] = to_nullable_string(df[c])

    # Publish_Year → nullable Int16
    df["Year-Of-Publication"] = coerce_year_to_int16(df["Year-Of-Publication"])
    
    #Drop ISBN columns
    isbn_cols = [c for c in df.columns if "isbn" in c.lower()]
    if isbn_cols:
        df = df.drop(columns=isbn_cols)

    # (Optional but useful): enforce dtypes explicitly for downstream Parquet
    dtype_map = {
        "title": "string",
        "Book-Author": "string",
        "Publisher": "string",
        "Year-Of-Publication": "Int16",
    }
    for c, dt in dtype_map.items():
        if dt == "string":
            df[c] = df[c].astype("string")
        else:
            df[c] = df[c].astype(dt)

    return df

def save_parquet(df: pd.DataFrame, path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Use a stable engine; 'pyarrow' preferred
    df.to_parquet(path, index=False)


amazon_clean = process_amazon(amazon)
save_parquet(amazon_clean, "parquet/amazon.parquet")


In [9]:
amazon_parquet = pd.read_parquet("parquet/amazon.parquet")
amazon_parquet.head()

,title,Book-Author,Year-Of-Publication,Publisher
0,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada
2,Decision in Normandy,Carlo D'Este,1991,HarperPerennial
3,Flu: The Story of the Great Influenza Pandemic of 1918 and the Search for the Virus That Caused It,Gina Bari Kolata,1999,Farrar Straus Giroux
4,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company


### Transformation for Goodreads

In [10]:
# Goodreads
import pandas as pd
import numpy as np

def extract_publish_year(s: pd.Series, pivot: int = 30) -> pd.Series:
    """
    Extract a 4-digit year from messy publishDate values.

    Handles:
      - MM/DD/YY with a century pivot (00–pivot-1 → 2000s, else 1900s)
      - Any 4-digit year in the text
      - Parsable dates (month names, ISO, etc.)
    Returns nullable Int16.
    """
    s = s.astype("string")

    # --- Case 1: exact MM/DD/YY -> resolve 2-digit year ---
    mask_mdyy = s.str.fullmatch(r"\s*\d{1,2}/\d{1,2}/\d{2}\s*")
    yy = pd.to_numeric(
        s.where(mask_mdyy).str.extract(r"(\d{2})\s*$")[0],
        errors="coerce"
    )
    # use NumPy to avoid pd.NA ambiguity
    yy_np = yy.to_numpy(dtype="float64")  # np.nan for missing
    year2_np = np.where(
        ~np.isnan(yy_np),
        np.where(yy_np >= pivot, 1900 + yy_np, 2000 + yy_np),
        np.nan
    )
    year_2dig = pd.Series(year2_np, index=s.index, dtype="Float64")

    # --- Case 2: any 4-digit year present ---
    year_4dig = pd.to_numeric(s.str.extract(r"(\d{4})")[0], errors="coerce").astype("Float64")

    # --- Case 3: general datetime parsing fallback ---
    dt = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)
    year_dt = dt.dt.year.astype("Float64")

    # Prefer: resolved 2-digit > explicit 4-digit > parsed datetime
    year = year_2dig.fillna(year_4dig).fillna(year_dt)
    return year.astype("Int16")

col = "publishDate"
pos = goodreads.columns.get_loc(col)
goodreads.insert(pos, "Publish_Year", extract_publish_year(goodreads[col], pivot=30))
goodreads.drop(columns=[col], inplace=True)


def parse_pages_to_int(s: pd.Series, dtype: str = "Int32") -> pd.Series:
    """
    Extract an integer page count from strings like "352 pages", "xvi + 250", "250".
    NA-safe and tolerant of non-strings.
    """
    import re
    import pandas as pd
    s = s.astype("string")  # gives you <NA> for missing

    def _one(tok):
        if pd.isna(tok):          # <- handle <NA>
            return pd.NA
        t = str(tok)
        m = re.search(r"\d{1,6}", t)
        if not m:
            return pd.NA
        val = int(m.group(0))
        return val if val > 0 else pd.NA

    return s.apply(_one).astype(dtype)


def process_goodreads(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    """
    Goodreads schema (input):
      isbn, title, author, rating, numRatings, language, genres, bookFormat,
      edition, pages, publisher, Publish_Year (Int16), price, ISBN_clean

    Steps:
      - Normalize text columns to pandas 'string' (uses to_nullable_string from earlier).
      - Replace isbn with ISBN_clean where available, then drop ISBN_clean.
      - price -> float64 (parse_price_to_float)
      - pages -> Int16 (parse_pages_to_int)
      - Keep Publish_Year as Int16 (already correct).
    """
    required = {
        "title","author","rating","numRatings","language","genres",
        "bookFormat","edition","pages","publisher","Publish_Year","price"
    }

    # Normalize key text columns (reuses your previously defined `to_nullable_string`)
    text_cols = ["title","author","language","genres","bookFormat","edition","pages","publisher","price"]
    for c in text_cols:
        df[c] = to_nullable_string(df[c])  # <- from earlier code

    
    #Drop ISBN columns
    isbn_cols = [c for c in df.columns if "isbn" in c.lower()]
    if isbn_cols:
        df = df.drop(columns=isbn_cols)
    
    # Price -> float64
    df["price"] = (df["price"].str.strip().replace({"": pd.NA})
      .pipe(pd.to_numeric, errors="coerce")
      .astype("Float32")
)

    # Pages -> nullable Int16 (safer upper bound than Int16)
    df["pages"] = parse_pages_to_int(df["pages"], dtype="Int16")

    # Dtypes you likely want to enforce for Parquet
    dtype_map = {
        "isbn": "string",
        "title": "string",
        "author": "string",
        "language": "string",
        "genres": "string",
        "bookFormat": "string",
        "edition": "string",
        "publisher": "string",
        "Publish_Year": "Int16",  # as given
        "rating": "float64",
        "numRatings": "Int64",    # make it nullable to avoid upcasts to float on NA
        "price": "float64",
        "pages": "Int32",
    }
    # Apply dtype map carefully (skip if already correct)
    for c, dt in dtype_map.items():
        if c in df.columns:
            try:
                df[c] = df[c].astype(dt)
            except TypeError:
                # If astype fails due to pandas version quirks, leave as-is (it’s already parsed)
                pass

    return df


def save_goodreads_parquet(df_in: pd.DataFrame, out_path: str | Path):
    """
    Process and write Goodreads to Parquet.
    Reuses your earlier `save_parquet` helper for consistent settings.
    """
    df_out = process_goodreads(df_in)
    save_parquet(df_out, out_path)  # <- uses the previously defined save_parquet
    return df_out

save_goodreads_parquet(process_goodreads(goodreads), "parquet/goodreads.parquet")


/var/folders/99/1mpdqqd52gx9ngr3lnl8wkzh0000gn/T/ipykernel_47547/2446054797.py:36: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dt = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)
/var/folders/99/1mpdqqd52gx9ngr3lnl8wkzh0000gn/T/ipykernel_47547/2446054797.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)


,title,author,rating,numRatings,language,genres,bookFormat,edition,pages,publisher,Publish_Year,price
0,The Hunger Games,Suzanne Collins,4.33,6376780,English,"['Young Adult', 'Fiction', 'Dystopia', 'Fantasy', 'Science Fiction', 'Romance', 'Adventure', 'Teen', 'Post Apocalyptic', 'Action']",Hardcover,First Edition,374,Scholastic Press,2008,5.09
1,Harry Potter and the Order of the Phoenix,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,2507623,English,"['Fantasy', 'Young Adult', 'Fiction', 'Magic', 'Childrens', 'Adventure', 'Audiobook', 'Middle Grade', 'Classics', 'Science Fiction Fantasy']",Paperback,US Edition,870,Scholastic Inc.,2004,7.38
2,To Kill a Mockingbird,Harper Lee,4.28,4501075,English,"['Classics', 'Fiction', 'Historical Fiction', 'School', 'Literature', 'Young Adult', 'Historical', 'Novels', 'Read For School', 'High School']",Paperback,<NA>,324,Harper Perennial Modern Classics,2006,NaN
3,Pride and Prejudice,"Jane Austen, Anna Quindlen (Introduction)",4.26,2998241,English,"['Classics', 'Fiction', 'Romance', 'Historical Fiction', 'Literature', 'Historical', 'Novels', 'Historical Romance', 'Classic Literature', 'Adult']",Paperback,"Modern Library Classics, USA / CAN",279,Modern Library,2000,NaN
4,Twilight,Stephenie Meyer,3.60,4964519,English,"['Young Adult', 'Fantasy', 'Romance', 'Vampires', 'Fiction', 'Paranormal', 'Paranormal Romance', 'Supernatural', 'Teen', 'Urban Fantasy']",Paperback,<NA>,501,"Little, Brown and Company",2006,2.10
...,...,...,...,...,...,...,...,...,...,...,...,...
52473,Fractured,Cheri Schmidt (Goodreads Author),4.00,871,English,"['Vampires', 'Paranormal', 'Young Adult', 'Romance', 'Fantasy', 'Paranormal Romance', 'Magic', 'Urban Fantasy', 'New Adult', 'Werewolves']",Nook,<NA>,<NA>,Cheri Schmidt,2011,NaN
52474,Anasazi,Emma Michaels,4.19,37,English,"['Mystery', 'Young Adult']",Paperback,First Edition,190,Bokheim Publishing,2011,NaN
52475,Marked,Kim Richardson (Goodreads Author),3.70,6674,English,"['Fantasy', 'Young Adult', 'Paranormal', 'Angels', 'Romance', 'Demons', 'Supernatural', 'Paranormal Romance', 'Urban Fantasy', 'Fiction']",Paperback,<NA>,280,CreateSpace,2011,7.37
52476,Wayward Son,"Tom Pollack (Goodreads Author), John Loftus (Goodreads Author), Jim Alves",3.85,238,English,"['Fiction', 'Mystery', 'Historical Fiction', 'Adventure', 'Christian Fiction', 'Historical', 'Religion', 'Suspense', 'Christian', 'Archaeology']",Paperback,1st edition,507,Cascada Productions,2011,2.86


In [11]:
goodreads_parquet=pd.read_parquet("parquet/goodreads.parquet")
goodreads_parquet.head(2)

,title,author,rating,numRatings,language,genres,bookFormat,edition,pages,publisher,Publish_Year,price
0,The Hunger Games,Suzanne Collins,4.33,6376780,English,"['Young Adult', 'Fiction', 'Dystopia', 'Fantasy', 'Science Fiction', 'Romance', 'Adventure', 'Teen', 'Post Apocalyptic', 'Action']",Hardcover,First Edition,374,Scholastic Press,2008,5.09
1,Harry Potter and the Order of the Phoenix,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,2507623,English,"['Fantasy', 'Young Adult', 'Fiction', 'Magic', 'Childrens', 'Adventure', 'Audiobook', 'Middle Grade', 'Classics', 'Science Fiction Fantasy']",Paperback,US Edition,870,Scholastic Inc.,2004,7.38


### Transformation for Metabooks

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path


def process_books_meta(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Normalize some text columns
    for c in ["title","Author_Name","Language","categories","Publisher","published_date","price"]:
        df[c] = to_nullable_string(df[c])

    # --- Year extraction: first 4-digit year from 'published_date' -> Int16
    years = df["published_date"].str.extract(r"(\d{4})", expand=False)
    df["Publish_Year"] = coerce_year_to_int16(years)
    df = df.drop(columns=["published_date"])

    # --- Price: clean decimal strings -> float (handle em dashes etc.)
    df["price"] = (
        df["price"]
          .str.replace("—", "", regex=False)   # remove em dash tokens
          .str.replace("-", "", regex=False)   # stray hyphens
          .str.strip()
          .replace({"": pd.NA})
          .pipe(pd.to_numeric, errors="coerce")
          .astype("Float32")
    )

    # --- Drop ALL ISBN-related columns
    isbn_cols = [c for c in df.columns if "isbn" in c.lower()]
    if isbn_cols:
        df = df.drop(columns=isbn_cols)

    # If it's already numeric, cast; if strings ever appear, coerce first.
    df["Page_Count"] = pd.to_numeric(df["Page_Count"], errors="coerce").round().astype("Int16")

    # Make rating_number nullable int to avoid float upcast on NAs
    df["rating_number"] = pd.to_numeric(df["rating_number"], errors="coerce").astype("Int64")

    # Enforce final dtypes you likely want in Parquet
    dtype_map = {
        "title": "string",
        "Author_Name": "string",
        "Language": "string",
        "categories": "string",
        "Publisher": "string",
        "average_rating": "float64",
        "rating_number": "Int64",
        "Publish_Year": "Int16",
        "Page_Count": "Int32",
        "price": "Float32",
    }
    for c, dt in dtype_map.items():
        if c in df.columns:
            df[c] = df[c].astype(dt)

    return df

def save_books_meta_parquet(df: pd.DataFrame, out_path: str | Path):
    clean = process_books_meta(df)
    save_parquet(clean, out_path)
    return clean

save_books_meta_parquet(metabooks, "parquet/metabooks.parquet")


,title,Author_Name,average_rating,rating_number,Language,categories,Publisher,Page_Count,price,Publish_Year
0,Alvis: The Story of the Red Triangle,Kenneth Day,4.1,3,English,"['Engineering', 'Transportation', 'Automotive']",Haynes Pubns,400,112.300003,<NA>
1,Siddhartha (Mondo Folktales),Hermann Hesse,4.6,1116,English,"['Literature', 'Fiction', 'Classics']",Audio Partners,<NA>,<NA>,<NA>
2,My Fair Billionaire (Harlequin Desire),Elizabeth Bevarly,4.3,17,English,"['Literature', 'Fiction', ""Women's Fiction""]",Harlequin Desire,192,3.95,<NA>
3,YE ARE GODS,Annalee Skarin,4.6,142,English,"['Religion', 'Spirituality', 'New Age']",DeVorss Publications,343,15.95,<NA>
4,The Calling of Emily Evans (Women of the West #1),Janette Oke,4.5,2730,English,"['Christian Books', 'Bibles', 'Literature', 'Fiction']",Sagebrush,<NA>,0.8,<NA>
...,...,...,...,...,...,...,...,...,...,...
535154,"The Penny-Pinching Prepper: Save More, Spend Less and Get Prepared for Any Disaster",Bernie Carr,4.6,37,English,"['Business', 'Money', 'Personal Finance']",Ulysses Press,296,12.95,2015
535155,The Essential Talmud,Adin Steinsaltz,4.6,203,English,"['Religion', 'Spirituality', 'Judaism']",Jason Aronson Inc,306,40.0,1994
535156,Making ADHD a Gift: Teaching Superman How to Fly,Robert E. Cimera,3.2,4,English,"['Education', 'Teaching', 'Schools']",R&L Education,168,2.79,2002
535157,Enterprise PowerShell Scripting Bootcamp: The fastest way to learn PowerShell scripting,Brenton J.W. Blawat,3.3,3,English,"['Computers', 'Technology', 'Networking', 'Cloud Computing']",Packt Publishing,238,48.990002,2017


In [13]:
metabooks_parquet=pd.read_parquet("parquet/metabooks.parquet")
metabooks_parquet.head()

,title,Author_Name,average_rating,rating_number,Language,categories,Publisher,Page_Count,price,Publish_Year
0,Alvis: The Story of the Red Triangle,Kenneth Day,4.1,3,English,"['Engineering', 'Transportation', 'Automotive']",Haynes Pubns,400,112.300003,<NA>
1,Siddhartha (Mondo Folktales),Hermann Hesse,4.6,1116,English,"['Literature', 'Fiction', 'Classics']",Audio Partners,<NA>,<NA>,<NA>
2,My Fair Billionaire (Harlequin Desire),Elizabeth Bevarly,4.3,17,English,"['Literature', 'Fiction', ""Women's Fiction""]",Harlequin Desire,192,3.95,<NA>
3,YE ARE GODS,Annalee Skarin,4.6,142,English,"['Religion', 'Spirituality', 'New Age']",DeVorss Publications,343,15.95,<NA>
4,The Calling of Emily Evans (Women of the West #1),Janette Oke,4.5,2730,English,"['Christian Books', 'Bibles', 'Literature', 'Fiction']",Sagebrush,<NA>,0.8,<NA>


In [15]:
display(metabooks_parquet.shape)
display(metabooks.shape)
display(amazon.shape)
display(amazon_parquet.shape)
display(goodreads.shape)
display(goodreads_parquet.shape)

(535159, 10)

(535159, 12)

(271360, 6)

(271360, 4)

(52478, 14)

(52478, 12)